## Installing Requered Libraries

In [1]:
%pip install langchain chromadb beautifulsoup4 sentence-transformers langchain-community
%pip install langchain-google-genai
%pip install --upgrade langsmith
%pip install python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Loading the API Keys and Environment Variables

In [2]:
import os
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["HF_API_KEY"] = os.getenv("HF_API_KEY")

# Set LangSmith project and HF API key
LANGSMITH_PROJECT = "Generative_AI2_RAG"
hf_api_key = os.getenv("HF_API_KEY")


## Fetching and Preprocessing the Web Article

In [3]:
import requests
from bs4 import BeautifulSoup

def fetch_article(url):
    """Fetches and processes the article from the given URL."""
    response = requests.get(url)
    if response.status_code != 200:
        raise Exception(f"Failed to fetch article. Status code: {response.status_code}")
    soup = BeautifulSoup(response.text, "html.parser")
    return soup.get_text(separator="\n", strip=True)

article_url = "https://www.coursera.org/articles/what-is-artificial-intelligence"
article_text = fetch_article(article_url)

# Save article content to a file
with open("article.txt", "w", encoding="utf-8") as file:
    file.write(article_text)

## Spliting the Document into Chunks and Embed Using ChromaDB

In [4]:
import shutil
import os

# Delete the existing 'rag_db' directory if it exists
if os.path.exists("rag_db"):
    shutil.rmtree("rag_db")
    print("Deleted existing 'rag_db' directory.")
else:
    print("'rag_db' directory does not exist.")


Deleted existing 'rag_db' directory.


In [5]:
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.vectorstores import Chroma
from langchain.embeddings import HuggingFaceEmbeddings

def process_document(file_path, chunk_size=300, chunk_overlap=50):
    """Loads the document, splits it into chunks, and embeds it."""
    loader = TextLoader(file_path, encoding="utf-8")
    docs = loader.load()
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    chunks = splitter.split_documents(docs)
    return chunks

chunks = process_document("article.txt")

# Ensure we have enough chunks for indexing
if len(chunks) < 50:
    raise ValueError(f"Not enough chunks. Only {len(chunks)} found. Add more content.")

# Initialize embeddings and create the Chroma vector store
embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L12-v2")
db = Chroma.from_documents(chunks, embedding, persist_directory="rag_db")
db.persist()



C:\Users\hshakademie7\AppData\Local\Temp\ipykernel_37504\1857312172.py:21: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L12-v2")
c:\Users\hshakademie7\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\hshakademie7\AppData\Local\Temp\ipykernel_37504\1857312172.py:23: LangChainDeprecationWarning: Since Chroma 0.4.x the manual persistence method is no longer s

## Setting Up LLM and Retriever for RAG

In [7]:
from langchain_google_genai.chat_models import ChatGoogleGenerativeAI
from langchain.chains import RetrievalQA
from langchain.retrievers.multi_query import MultiQueryRetriever

# Initialize the LLM (Gemini 2.0)
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)

# Create MultiQueryRetriever with Chroma vector store
multi_retriever = MultiQueryRetriever.from_llm(
    retriever=db.as_retriever(),
    llm=llm
)

# Default QA Chain using MultiQueryRetriever
default_chain = RetrievalQA.from_chain_type(llm=llm, retriever=multi_retriever)


## Adding  Conversation Memory for Multi-turn Dialogs


In [15]:
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.chains.combine_documents.stuff import StuffDocumentsChain

# Setup memory
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

# Define a custom prompt template for a more conversational approach
custom_template = """You are a helpful AI tutor. Answer the question using only the context below.And only answer in German.
Context:
{context}

Question:
{question}

Answer:"""

custom_prompt = PromptTemplate(input_variables=["context", "question"], template=custom_template)
custom_llm_chain = LLMChain(llm=llm, prompt=custom_prompt)
custom_chain_type = StuffDocumentsChain(llm_chain=custom_llm_chain, document_variable_name="context")

# Custom chain for retrieval with memory
custom_chain = RetrievalQA(combine_documents_chain=custom_chain_type, retriever=multi_retriever, memory=memory)


## Evaluation with Sample Questions


In [17]:
questions = [
    "What is artificial intelligence?",
    "What are the disadvantages of using AI?",
    "What are self-aware machines?",
    "How is AI used in everyday life?",
    "What skills are needed to work in AI?",
    "what is was the AI Rule in 2020?",
    "what is the current population in germany?",
    "what is the future of AI?",
    "how was Ai discovered?",
]

from IPython.display import Markdown

def evaluate_questions(questions, default_chain, custom_chain):
    """Evaluates questions using both default and custom chains."""
    for i, question in enumerate(questions, 1):
        relevant_docs = multi_retriever.get_relevant_documents(question)
        if not relevant_docs:
            custom_answer = "I don't know."
        else:
            custom_answer = custom_chain.run(question)
        default_answer = default_chain.run(question)
        
        display(Markdown(f"""
### Q{i}: {question}
**🔹 Default Answer:**  
{default_answer}

**🔸 Custom Answer:**  
{custom_answer}

---
"""))

evaluate_questions(questions, default_chain, custom_chain)



os.environ["LANGSMITH_TRACING"] = "true"


### Q1: What is artificial intelligence?
**🔹 Default Answer:**  
Artificial intelligence (AI) is the theory and development of computer systems capable of performing tasks that historically required human intelligence, such as recognizing speech, making decisions, and identifying patterns.

**🔸 Custom Answer:**  
Künstliche Intelligenz (KI) bezieht sich auf die Theorie und Entwicklung von Computersystemen, die in der Lage sind, Aufgaben auszuführen, die historisch menschliche Intelligenz erforderten, wie z. B. Spracherkennung, Entscheidungsfindung und Mustererkennung.

---



### Q2: What are the disadvantages of using AI?
**🔹 Default Answer:**  
Based on the provided text, here are some potential dangers of using AI:

*   Potential for bias or discrimination as a result of the data set on which the AI is trained.
*   Possible cybersecurity concerns.

**🔸 Custom Answer:**  
Einige der möglichen Gefahren von KI sind:

*   Potenzial für Voreingenommenheit oder Diskriminierung aufgrund des Datensatzes, mit dem die KI trainiert wird.
*   Mögliche Bedenken hinsichtlich der Cybersicherheit.

---



### Q3: What are self-aware machines?
**🔹 Default Answer:**  
Self-aware machines are the most advanced type of AI, possessing an understanding of the world, others, and themselves. This is often what people refer to when talking about achieving AGI (artificial general intelligence), but it is currently a distant prospect.

**🔸 Custom Answer:**  
Selbstbewusste Maschinen sind die theoretisch fortschrittlichste Art von KI und würden ein Verständnis der Welt, anderer und ihrer selbst besitzen.

---



### Q4: How is AI used in everyday life?
**🔹 Default Answer:**  
AI is used in everyday life in many ways, including:

*   Making song recommendations
*   Identifying the fastest way to travel to a destination
*   Translating text from one language to another
*   Personalization within digital services and products

**🔸 Custom Answer:**  
KI wird im Alltag für Aufgaben wie Musikempfehlungen, die Ermittlung des schnellsten Reisewegs zu einem Ziel oder die Übersetzung von Texten verwendet. Beispiele sind ChatGPT und maschinelles Lernen.

---



### Q5: What skills are needed to work in AI?
**🔹 Default Answer:**  
The text mentions enrolling in the "IBM AI Foundations for Everyone Specialization" to build job-ready AI skills. It also suggests taking DeepLearning.AI's "AI For Everyone" course to learn what AI can realistically do and how to apply it to problems.

**🔸 Custom Answer:**  
Um in der KI zu arbeiten, sind Fähigkeiten erforderlich, die es Computersystemen ermöglichen, komplexe Aufgaben auszuführen, die historisch menschliche Intelligenz erforderten, wie z. B. Schlussfolgern, Entscheidungen treffen oder Probleme lösen. Die IBM AI Foundations for Everyone Specialization kann Ihnen helfen, diese Fähigkeiten zu erwerben.

---



### Q6: what is was the AI Rule in 2020?
**🔹 Default Answer:**  
I am sorry, but I cannot provide information about any AI rules specifically from 2020. The text does not mention any AI rules.

**🔸 Custom Answer:**  
Der Kontext enthält keine Informationen über eine KI-Regel im Jahr 2020.

---



### Q7: what is the current population in germany?
**🔹 Default Answer:**  
I am sorry, I cannot provide you with the current population of Germany. This document is focused on defining artificial intelligence, its potential benefits and dangers, and examples of AI in use today.

**🔸 Custom Answer:**  
Die zur Verfügung gestellten Informationen enthalten keine Angaben zur aktuellen Bevölkerung Deutschlands.

---



### Q8: what is the future of AI?
**🔹 Default Answer:**  
AI has a wide array of applications with the potential to transform how we work and live, including self-driving cars, virtual assistants, and wearable devices. AI is also being employed in different industries like finance. Some common examples of AI in use today include ChatGPT and machine learning models.

**🔸 Custom Answer:**  
Der Kontext enthält keine Informationen über die Zukunft der KI.

---



### Q9: how was Ai discovered?
**🔹 Default Answer:**  
I am sorry, but the context provided does not contain information on how AI was discovered.

**🔸 Custom Answer:**  
Künstliche Intelligenz (KI) bezieht sich auf Computersysteme, die in der Lage sind, komplexe Aufgaben auszuführen, die historisch nur ein Mensch ausführen konnte, wie z. B. Schlussfolgern, Entscheidungen treffen oder Probleme lösen.

---
